# Contraction Hierarchy Bidirectional Search Demo

This notebook reproduces the logic from the Python prototype used to validate the C++ query engine. It loads the shortcut and edge metadata tables, rebuilds the adjacency lists, and exercises the hierarchy-aware bidirectional Dijkstra implementation.

## Load Edge Metadata

Read the precomputed edge CSV to recover incoming cells and hierarchy resolution used as constraints during search.

In [ ]:
import pandas as pd

district = "Somerset" #"Burnaby"
#district ="Burnaby"

# Update this path if you want to work with a different edge metadata snapshot.
edges_PATH = f"../../spark-shortest-path/data/{district}_driving_simplified_edges_with_h3.csv"

# Load the edge table and retain it with the edge id as index for fast lookups.
edges = pd.read_csv(edges_PATH)
edges_df = edges.set_index('id')
print(f"Loaded {len(edges):,} all edges")
edges.head()

Loaded 5,900 all edges


,source,target,length,maxspeed,geometry,highway,cost,incoming_cell,outgoing_cell,lca_res,id
0,167790704,167862530,80.287,60.0,LINESTRING (-84.60797882080078 37.091793060302...,secondary,4.817220,645224977384028320,645224977383611141,8,0
1,167790704,167790717,102.097,60.0,LINESTRING (-84.60797882080078 37.091793060302...,secondary,6.125820,645224977383614665,645224977383611141,10,1
2,167790704,167825885,323.104,50.0,LINESTRING (-84.60797882080078 37.091793060302...,tertiary,23.263488,645224977384658840,645224977383611141,8,2
3,167790717,167790719,111.432,60.0,LINESTRING (-84.60697937011719 37.091552734375...,secondary,6.685920,645224977383653531,645224977383614665,9,3
4,167790717,167862553,84.317,30.0,LINESTRING (-84.60697937011719 37.091552734375...,residential,10.118040,645224977383600429,645224977383614665,10,4


## Load Shortcut Parquet

Read the precomputed shortcut catalog, enforce numeric types, and inspect the first few rows.

In [ ]:

PARQUET_PATH = f"../../spark-shortest-path/output/{district}_shortcuts_final"

# Load shortcuts and normalise schema for downstream computations.
df = pd.read_parquet(PARQUET_PATH)

# Ensure deterministic types for the routing algorithm.
df['incoming_edge'] = df['incoming_edge'].astype(int)
df['outgoing_edge'] = df['outgoing_edge'].astype(int)
df['via_edge'] = df['via_edge'].fillna(0).astype(int)
df['inside'] = df['inside'].astype(int)
df['cost'] = df['cost'].astype(float)

print(f"Loaded {len(df):,} shortcut edges")
df.head()

Loaded 99,413 shortcut edges


,incoming_edge,outgoing_edge,cost,via_edge,inside,cell
0,2310,447,31.018371,2311,0,0
1,3053,3055,13.830667,3049,0,0
2,3365,3354,14.861838,3362,0,0
3,3354,3365,18.916200,3362,0,0
4,4469,4471,6.867050,4482,0,0


In [ ]:
# Reload shortcuts after regeneration
df = pd.read_parquet(PARQUET_PATH)

# Ensure deterministic types for the routing algorithm.
df['incoming_edge'] = df['incoming_edge'].astype(int)
df['outgoing_edge'] = df['outgoing_edge'].astype(int)
df['via_edge'] = df['via_edge'].fillna(0).astype(int)
df['inside'] = df['inside'].astype(int)
df['cost'] = df['cost'].astype(float)

print(f"Reloaded {len(df):,} shortcut edges")

# Filter for specific edge pair using pandas syntax
result = df[(df["incoming_edge"] == 415) & (df["outgoing_edge"] == 5253)]
if result.empty:
    print("\n415 → 5253 does NOT exist as a direct shortcut (was filtered out as invalid)")
    print("\nPath should be found via intermediate nodes:")
    print("  415 → 1858 → 538 → 5253")
else:
    print(f"\n415 → 5253 exists with via_edge={int(result.iloc[0]['via_edge'])}")

result

,incoming_edge,outgoing_edge,cost,via_edge,inside,cell


In [ ]:
# Test the same query that's failing in C++
# If the problem is in the bidirectional search, we should see it here too
TEST_SOURCE = 407  # Replace with actual source from your query
TEST_TARGET = 415  # Replace with actual target from your query

print(f"Testing query: {TEST_SOURCE} -> {TEST_TARGET}")
highcell = high_cell(TEST_SOURCE, TEST_TARGET, edges_df)
highcellres = high_cell_resolution(highcell)
target_cost = edges_df.loc[TEST_TARGET].get('length', 0.0) if TEST_TARGET in edges_df.index else 0.0

distance, path = bidijkstra(TEST_SOURCE, TEST_TARGET, highcell, highcellres, target_cost=target_cost)

if distance == -1:
    print("No path found.")
else:
    print(f"Distance: {distance}")
    print(f"Shortcut path ({len(path)} edges): {path}")
    
    # Check each consecutive pair to see if shortcuts exist
    print("\n--- Validating shortcut path ---")
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        try:
            sc = shortcut_lookup.loc[(u, v)]
            print(f"✓ {u} -> {v}: via_edge={int(sc['via_edge'])}, cost={sc['cost']:.2f}")
        except KeyError:
            print(f"✗ {u} -> {v}: NO SHORTCUT FOUND!")
    
    print("\n--- Expanded path ---")
    base_path = expand_shortcut_path(path, shortcut_lookup)
    print(f"Base edges ({len(base_path)}): {base_path}")

In [23]:
#9219 -> 9214
df[(df["incoming_edge"] == 9219) & (df["outgoing_edge"] == 9214)]

,incoming_edge,outgoing_edge,cost,via_edge,inside,cell
1541996,9219,9214,4.61604,9214,-1,613208529485955071


In [13]:
df[(df["incoming_edge"] == 405) & (df["outgoing_edge"] == 408)]

,incoming_edge,outgoing_edge,cost,via_edge,inside,cell
204868,405,408,3.405333,408,-1,622706979251388415


In [14]:
df[(df["incoming_edge"] == 408) & (df["outgoing_edge"] == 415)]

,incoming_edge,outgoing_edge,cost,via_edge,inside,cell
203595,408,415,3.683867,415,1,622706979251388415


In [ ]:
#SOURCE_EDGE = 407
#TARGET_EDGE = 415
#407 → 1187 → 405 → 408 → 415 → 2770 → 5253 → 4388 → 7 → 5

## Build Forward and Backward Adjacency Lists

Convert the shortcut table into adjacency dictionaries that the bidirectional search consumes.

In [ ]:
import heapq

# Build Adjacency Lists
# fwd_adj: u -> [(v, cost, inside), ...]
# bwd_adj: v -> [(u, cost, inside), ...]

print("Building Graph...")
fwd_adj = {}
bwd_adj = {}

# We can iterate over the dataframe or convert to records for speed
records = df[['incoming_edge', 'outgoing_edge', 'cost', 'inside','cell']].to_dict('records')

for row in records:
    u = row['incoming_edge']
    v = row['outgoing_edge']
    c = row['cost']
    inside = row['inside']
    cell = row['cell']
    
    if u not in fwd_adj: fwd_adj[u] = []
    fwd_adj[u].append((v, c, cell, inside))
    
    if v not in bwd_adj: bwd_adj[v] = []
    bwd_adj[v].append((u, c, cell, inside))

shortcut_lookup = df.set_index(['incoming_edge', 'outgoing_edge'])

print(f"Graph built. Nodes with outgoing edges: {len(fwd_adj)}, Nodes with incoming edges: {len(bwd_adj)}")
print(f"\n✓ Graph ready for queries")


Building Graph...
Graph built. Nodes with outgoing edges: 34965, Nodes with incoming edges: 34965
Graph built. Nodes with outgoing edges: 34965, Nodes with incoming edges: 34965


In [ ]:
# Test query: 415 → 5252
print("Testing query: 415 → 5252")
print("="*60)

SOURCE_EDGE = 415
TARGET_EDGE = 5252

# First check if path exists in graph (ignoring hierarchy)
print("\nChecking raw connectivity (BFS):")
from collections import deque
visited = {SOURCE_EDGE}
parent = {SOURCE_EDGE: None}
queue = deque([SOURCE_EDGE])
found = False

while queue and not found:
    u = queue.popleft()
    if u == TARGET_EDGE:
        found = True
        break
    for v, cost, cell, inside in fwd_adj.get(u, []):
        if v not in visited:
            visited.add(v)
            parent[v] = u
            queue.append(v)

if found:
    path_raw = []
    node = TARGET_EDGE
    while node is not None:
        path_raw.append(node)
        node = parent[node]
    path_raw.reverse()
    print(f"✓ BFS path exists: {' → '.join(map(str, path_raw))}")
    
    # Show edge types
    for i in range(len(path_raw)-1):
        u, v = path_raw[i], path_raw[i+1]
        edge_info = [(cost, cell, inside) for dst, cost, cell, inside in fwd_adj.get(u, []) if dst == v][0]
        print(f"  {u} → {v}: inside={edge_info[2]}")
else:
    print("✗ No path in graph at all!")

# Now test with hierarchy constraints
print("\n" + "="*60)
print("Testing with hierarchy constraints:")

highcell = high_cell(SOURCE_EDGE, TARGET_EDGE, edges_df)
highcellres = high_cell_resolution(highcell)
target_cost = edges_df.loc[TARGET_EDGE].get('length', 0.0) if TARGET_EDGE in edges_df.index else 0.0

print(f"High cell: {highcell}, resolution: {highcellres}")
print(f"Target edge cost: {target_cost:.2f}")

distance, path = bidijkstra(SOURCE_EDGE, TARGET_EDGE, highcell, highcellres, target_cost=target_cost)

if distance == -1:
    print("\n✗ No path found by bidirectional Dijkstra!")
    print("\nProblem: The path requires edges with inside=0 or inside=-1,")
    print("but forward search only accepts inside=1 (upward edges).")
    print("\nThis is a fundamental issue with the hierarchy constraints.")
else:
    print(f"\n✓ Path found!")
    print(f"Distance: {distance:.2f}")
    print(f"Shortcut path ({len(path)} edges): {path}")

## H3 Utilities

Utility helpers to compute ancestor cells and verify that shortcuts stay within the allowable hierarchy.

## Bidirectional Dijkstra

Implementation of the hierarchy-aware search that processes upward shortcuts forward and downward (plus lateral-at-top) shortcuts backward.

In [26]:
import h3


def _find_ancestor_impl(cell: int, res: int) -> int:
    """Return the ancestor of ``cell`` at resolution ``res`` (or 0 if invalid)."""
    if cell == 0 or res < 0:
        return 0
    if res > h3.get_resolution(h3.int_to_str(cell)):
        return cell
    return h3.str_to_int(h3.cell_to_parent(h3.int_to_str(cell), res))

def find_lca(cell1: int, cell2: int) -> int:
    """Compute the lowest common ancestor between two H3 cells."""
    if cell1 == 0 or cell2 == 0:
        return 0
    cell1_res = h3.get_resolution(h3.int_to_str(cell1))
    cell2_res = h3.get_resolution(h3.int_to_str(cell2))
    lca_res = min(cell1_res, cell2_res)
    while lca_res >= 0:
        if h3.cell_to_parent(h3.int_to_str(cell1), lca_res) == h3.cell_to_parent(h3.int_to_str(cell2), lca_res):
            return h3.str_to_int(h3.cell_to_parent(h3.int_to_str(cell1), lca_res))
        lca_res -= 1
    return 0

def high_cell(source_edge_id: int, destination_edge_id: int, edges_df: pd.DataFrame) -> int:
    """Return the highest common ancestor cell for the two edge endpoints."""
    s_cell = edges_df.loc[source_edge_id]["incoming_cell"]
    s_res = edges_df.loc[source_edge_id]["lca_res"]
    d_cell = edges_df.loc[destination_edge_id]["incoming_cell"]
    d_res = edges_df.loc[destination_edge_id]["lca_res"]
    source_cell = _find_ancestor_impl(s_cell, res=s_res)
    destination_cell = _find_ancestor_impl(d_cell, res=d_res)
    return find_lca(source_cell, destination_cell)

def high_cell_resolution(highcell: int) -> int:
    if highcell == 0:
        return -1
    return h3.get_resolution(h3.int_to_str(highcell))

def parent_check(child_cell: int, parent_cell: int, parent_res: int) -> bool:
    """Verify that ``child_cell`` lies within ``parent_cell`` at ``parent_res``."""
    if parent_cell == 0:
        return True
    if child_cell == 0:
        return False
    child_res = h3.get_resolution(h3.int_to_str(child_cell))
    if parent_res > child_res:
        return False
    derived_parent = _find_ancestor_impl(child_cell, parent_res)
    return derived_parent == parent_cell

In [27]:
def bidijkstra(source: int, target: int, highcell: int, highcellres: int, target_cost: float = 0.0):
    """Bidirectional Dijkstra constrained by the CH hierarchy.
    
    Parameters
    ----------
    source : int
        Source edge id.
    target : int
        Target edge id.
    highcell : int
        Highest common ancestor H3 cell.
    highcellres : int
        Resolution of the highest common ancestor cell.
    target_cost : float, optional
        Cost/weight of the destination edge to include in the total path cost.
        Default is 0.0 (not included).
    
    Returns
    -------
    tuple[float, list[int]]
        (total_cost, path) where total_cost includes target_cost if provided.
    """
    if source == target:
        return target_cost, [source]

    dist_fwd = {source: 0.0}
    dist_bwd = {target: 0.0}
    prev_fwd = {source: None}
    prev_bwd = {target: None}

    pq_fwd = [(0.0, source)]
    pq_bwd = [(0.0, target)]

    best_cost = float("inf")
    meeting_node = None

    while pq_fwd or pq_bwd:
        if pq_fwd:
            d, u = heapq.heappop(pq_fwd)
            if d > dist_fwd.get(u, float("inf")):
                continue

            if u in dist_bwd:
                total = d + dist_bwd[u]
                if total < best_cost:
                    best_cost = total
                    meeting_node = u

            if d >= best_cost:
                continue

            for v, cost, cell, inside in fwd_adj.get(u, []):
                if inside != 1 or not parent_check(cell, highcell, highcellres):
                    continue
                new_dist = d + cost
                if new_dist < dist_fwd.get(v, float("inf")):
                    dist_fwd[v] = new_dist
                    prev_fwd[v] = u
                    heapq.heappush(pq_fwd, (new_dist, v))

                    if v in dist_bwd:
                        total = new_dist + dist_bwd[v]
                        if total < best_cost:
                            best_cost = total
                            meeting_node = v

        if pq_bwd:
            d, v = heapq.heappop(pq_bwd)
            if d > dist_bwd.get(v, float("inf")):
                continue

            if v in dist_fwd:
                total = d + dist_fwd[v]
                if total < best_cost:
                    best_cost = total
                    meeting_node = v

            if d >= best_cost:
                continue

            for u, cost, cell, inside in bwd_adj.get(v, []):
                allow_lateral = highcell != 0 and inside == 0 and cell == highcell
                if inside not in (-1, 0) or (inside == 0 and not allow_lateral):
                    continue
                if not parent_check(cell, highcell, highcellres):
                    continue
                new_dist = d + cost
                if new_dist < dist_bwd.get(u, float("inf")):
                    dist_bwd[u] = new_dist
                    prev_bwd[u] = v
                    heapq.heappush(pq_bwd, (new_dist, u))

                    if u in dist_fwd:
                        total = new_dist + dist_fwd[u]
                        if total < best_cost:
                            best_cost = total
                            meeting_node = u

        if best_cost < float("inf"):
            if (
                pq_fwd
                and pq_fwd[0][0] >= best_cost
                and pq_bwd
                and pq_bwd[0][0] >= best_cost
            ):
                break

    if meeting_node is None:
        return -1, []

    forward_path = []
    node = meeting_node
    while node is not None:
        forward_path.append(node)
        node = prev_fwd.get(node)
    forward_path.reverse()

    backward_path = []
    node = prev_bwd.get(meeting_node)
    while node is not None:
        backward_path.append(node)
        node = prev_bwd.get(node)

    path = forward_path + backward_path
    # Add target_cost to the final path cost
    final_cost = best_cost + target_cost
    return final_cost, path

In [ ]:
def expand_shortcut_path(shortcut_path, shortcuts_df):
    """Expand a list of shortcut edge ids into the underlying base edge ids.

    The path is a sequence of edges where consecutive pairs (u, v) represent
    shortcuts. Each shortcut may have a via_edge that itself needs expansion.

    Parameters
    ----------
    shortcut_path : list[int]
        Ordered sequence of edge ids returned by the bidirectional search.
    shortcuts_df : pandas.DataFrame
        Shortcut table indexed by ``(incoming_edge, outgoing_edge)`` with column ``via_edge``.

    Returns
    -------
    list[int]
        Sequence of base edge ids after recursively expanding all shortcuts.
    """
    if not shortcut_path:
        return []
    if len(shortcut_path) == 1:
        return [int(shortcut_path[0])]
    
    def expand_pair(u, v, visited=None):
        """Recursively expand a shortcut (u -> v) into base edges."""
        if visited is None:
            visited = set()
        pair = (u, v)
        if pair in visited:
            return [u, v]  # Cycle detected
        visited.add(pair)
        
        try:
            row = shortcuts_df.loc[(u, v)]
        except KeyError:
            # No shortcut entry, these are consecutive base edges
            return [u, v]
        
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        
        via = row.get('via_edge', 0)
        via = 0 if pd.isna(via) else int(via)
        
        if via == 0 or via == u or via == v:
            # Base edge (no intermediate or via equals source/target), just return the pair
            return [u, v]
        
        # Recursively expand: u -> via and via -> v
        left = expand_pair(u, via, visited.copy())
        right = expand_pair(via, v, visited.copy())
        
        # Merge, avoiding duplicate at the junction
        if right and right[0] == left[-1]:
            return left + right[1:]
        return left + right
    
    # Expand each consecutive pair and merge
    base_edges = []
    for u, v in zip(shortcut_path, shortcut_path[1:]):
        expanded = expand_pair(int(u), int(v))
        for e in expanded:
            if not base_edges or base_edges[-1] != e:
                base_edges.append(e)
    
    return base_edges

## Expand Shortcuts to Base Edges

Translate the CH-level path into the underlying edge ids using the `via_edge` mapping.

In [32]:
import random
from time import perf_counter

# Sample one pair (or swap to random sampling) and run the bidirectional search.
all_nodes = list(set(df['incoming_edge'].unique()) | set(df['outgoing_edge'].unique()))
source = random.choice(all_nodes)  # for stochastic testing
target = random.choice(all_nodes)  # for stochastic testing

print(f"Query: {source} -> {target}")
highcell = high_cell(source, target, edges_df)
highcellres = high_cell_resolution(highcell)

# Get the cost of the destination edge from the edges table
# Adjust 'length' to the actual column name for edge weight/cost in your edges_df
target_cost = edges_df.loc[target].get('length', 0.0) if target in edges_df.index else 0.0

t_start = perf_counter()

distance, path = bidijkstra(source, target, highcell, highcellres, target_cost=target_cost)
elapsed = perf_counter() - t_start

if distance == -1:
    print("No path found.")
else:
    print(f"Distance (including destination edge): {distance}")
    print(f"Destination edge cost: {target_cost}")
    print(f"Path length: {len(path)} nodes")
    base_path = expand_shortcut_path(path, shortcut_lookup)
    print(f"Expanded base edge path: {base_path}")
    print(f"Shortcut path: {path}")

print(f"Runtime: {elapsed * 1000:.2f} ms")

Query: 4547 -> 24023
Distance (including destination edge): 146.61783666666668
Destination edge cost: 9.479
Path length: 7 nodes
Expanded base edge path: [4547, 4550, 25004, 25910, 30827, 3929, 24023]
Shortcut path: [np.int64(4547), 4550, 25004, 25910, 30827, 3929, np.int64(24023)]
Runtime: 153.07 ms


In [30]:
shortcut_path = path
for current_edge, next_edge in zip(shortcut_path, shortcut_path[1:]):
    print(current_edge, next_edge)
    print(shortcut_lookup.loc[(current_edge, next_edge)].apply(int))
    print("----------------------")

13790 13786
cost                         4
via_edge                 13786
inside                       1
cell        622215728747872256
Name: (13790, 13786), dtype: int64
----------------------
13786 7734
cost                         7
via_edge                  7734
inside                       1
cell        617712129120665600
Name: (13786, 7734), dtype: int64
----------------------
7734 8818
cost                         9
via_edge                  8818
inside                       1
cell        608704929871167488
Name: (7734, 8818), dtype: int64
----------------------
8818 16414
cost                        52
via_edge                   191
inside                       1
cell        599697731186851840
Name: (8818, 16414), dtype: int64
----------------------
16414 8353
cost                         9
via_edge                  8353
inside                      -1
cell        613208497397432320
Name: (16414, 8353), dtype: int64
----------------------


In [31]:
(current_edge, next_edge) = (17742, 24667)

shortcut_lookup.loc[(current_edge, next_edge)].apply(int).reset_index()

,index,17742
,,24667
0,cost,35
1,via_edge,24667
2,inside,1
3,cell,608704929619509248
